# Field & Soil Exploratory Data Analysis

**Assignment 03** | Agricultural Data Analysis

This notebook performs exploratory data analysis on field and soil data.

In [ ]:
# Setup and imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set style
plt.style.use('seaborn-v0_8-whitegrid')

# Earth tone color palette (balanced agricultural theme)
EARTH_TONES = [
    '#2E8B57',  # Sea Green
    '#DAA520',  # Goldenrod
    '#8B4513',  # Saddle Brown
    '#6B8E23',  # Olive Drab
    '#CD853F',  # Peru
    '#556B2F',  # Dark Olive Green
    '#A0522D',  # Sienna
    '#9ACD32',  # Yellow Green
    '#B8860B',  # Dark Goldenrod
    '#808000',  # Olive
]
sns.set_palette(EARTH_TONES)
sns.set_theme(style='whitegrid')

# Output directory
OUTPUT_DIR = 'data/assignment-03'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

## 1. Load Data

In [ ]:
# Load field summary data
df = pd.read_csv('data/assignment-02/field_summary.csv')
print(f"Fields: {len(df)}")
print(f"Columns: {list(df.columns)}")

## 2. Data Overview

In [ ]:
# Summary statistics
soil_properties = ['organic_matter', 'ph', 'clay_pct', 'sand_pct', 
                    'silt_pct', 'water_capacity', 'cec', 'bulk_density']

summary = df[soil_properties].describe().round(2)
summary.to_csv(f'{OUTPUT_DIR}/eda-summary.csv')
print("Summary statistics saved to eda-summary.csv\n")
print(summary)

In [ ]:
# Crop distribution
print("\nCrop distribution:")
print(df['crop_name'].value_counts())
print(f"\nTotal acres: {df['area_acres'].sum():,.1f}")

## 3. Distribution Visualizations

In [ ]:
# Distribution histograms
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

prop_labels = {
    'organic_matter': 'Organic Matter (%)',
    'ph': 'pH',
    'clay_pct': 'Clay (%)',
    'sand_pct': 'Sand (%)',
    'silt_pct': 'Silt (%)',
    'water_capacity': 'Water Capacity',
    'cec': 'CEC (meq/100g)',
    'bulk_density': 'Bulk Density'
}

for i, prop in enumerate(soil_properties):
    ax = axes[i]
    ax.hist(df[prop].dropna(), bins=15, edgecolor='black', alpha=0.7)
    ax.set_xlabel(prop_labels.get(prop, prop))
    ax.set_ylabel('Frequency')
    ax.axvline(df[prop].mean(), color='red', linestyle='--', label=f"Mean: {df[prop].mean():.2f}")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/soil-distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR}/soil-distributions.png")

## 4. Correlation Analysis

In [ ]:
# Correlation matrix
corr_matrix = df[soil_properties].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='BrBG', center=0, 
            fmt='.2f', square=True, linewidths=0.5)
plt.title('Soil Properties Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/correlation-matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR}/correlation-matrix.png")

## 5. Soil Properties by Crop Type

In [ ]:
# Box plots by crop type
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

crop_props = ['organic_matter', 'ph', 'clay_pct', 'cec', 'bulk_density', 'water_capacity']

for i, prop in enumerate(crop_props):
    ax = axes[i]
    sns.boxplot(data=df, x='crop_name', y=prop, ax=ax, palette=EARTH_TONES)
    ax.set_xlabel('Crop Type')
    ax.set_ylabel(prop_labels.get(prop, prop))
    ax.set_title(f'{prop_labels.get(prop, prop)} by Crop')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/soil-by-crop.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR}/soil-by-crop.png")

## 6. Drainage Class Analysis

In [ ]:
# Drainage class distribution
plt.figure(figsize=(12, 5))

drainage_counts = df['drainage_class'].value_counts()
colors = sns.color_palette(EARTH_TONES[:len(drainage_counts)])

plt.subplot(1, 2, 1)
drainage_counts.plot(kind='bar', color=colors, edgecolor='black')
plt.xlabel('Drainage Class')
plt.ylabel('Number of Fields')
plt.title('Fields by Drainage Class')
plt.xticks(rotation=45, ha='right')

plt.subplot(1, 2, 2)
drainage_area = df.groupby('drainage_class')['area_acres'].sum().sort_values(ascending=False)
drainage_area.plot(kind='bar', color=colors, edgecolor='black')
plt.xlabel('Drainage Class')
plt.ylabel('Total Acres')
plt.title('Area by Drainage Class')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/drainage-analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR}/drainage-analysis.png")

## 7. Key Findings Summary

In [ ]:
# Print key findings
print("=" * 60)
print("KEY FINDINGS")
print("=" * 60)

print(f"\n📊 Dataset: {len(df)} fields")
print(f"\n🌾 Crops: {dict(df['crop_name'].value_counts())}")
print(f"\n📈 Soil Property Ranges:")
print(f"   Organic Matter: {df['organic_matter'].min():.1f}% - {df['organic_matter'].max():.1f}% (avg: {df['organic_matter'].mean():.1f}%)")
print(f"   pH: {df['ph'].min():.1f} - {df['ph'].max():.1f} (avg: {df['ph'].mean():.1f})")
print(f"   CEC: {df['cec'].min():.1f} - {df['cec'].max():.1f} meq/100g (avg: {df['cec'].mean():.1f})")

print(f"\n💧 Drainage Distribution:")
for dc, count in df['drainage_class'].value_counts().items():
    acres = df[df['drainage_class'] == dc]['area_acres'].sum()
    print(f"   {dc}: {count} fields ({acres:,.0f} acres)")

print(f"\n🔗 Strongest Correlations:")
corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))

corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
for prop1, prop2, corr in corr_pairs[:5]:
    print(f"   {prop1} vs {prop2}: {corr:.2f}")

In [ ]:
# Save analysis dataset and print completion
df.to_csv(f'{OUTPUT_DIR}/fields_with_soil_analysis.csv', index=False)
print(f"\n✅ Analysis complete!")
print(f"   Output files saved to: {OUTPUT_DIR}/")
print(f"   - eda-summary.csv")
print(f"   - soil-distributions.png")
print(f"   - correlation-matrix.png")
print(f"   - soil-by-crop.png")
print(f"   - drainage-analysis.png")
print(f"   - fields_with_soil_analysis.csv")